In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
import os
import pickle
# import cv2
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import gymnasium as gym
from torch.utils.data import Dataset, DataLoader
import math

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class VAE(nn.Module):
    def __init__(self, latent_dim=512):
        super(VAE, self).__init__()
        self.latent_dim = latent_dim  # store it

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1),  # 64x64 -> 32x32
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),  # 32x32 -> 16x16
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1),  # 16x16 -> 8x8
            nn.ReLU(),
            nn.Conv2d(128, 256, 4, stride=2, padding=1),  # 8x8 -> 4x4
            nn.ReLU()
        )

        # Latent vectors mu and logvar
        self.fc_mu = nn.Linear(256*4*4, latent_dim)
        self.fc_logvar = nn.Linear(256*4*4, latent_dim)

        # Decoder fully connected
        self.fc_dec = nn.Linear(latent_dim, 256*4*4)

        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),  # 4x4 -> 8x8
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),   # 8x8 -> 16x16
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),    # 16x16 -> 32x32
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1),     # 32x32 -> 64x64
            nn.Sigmoid()  # output in [0,1]
        )

    def encode(self, x):
        x = self.encoder(x)
        x = x.view(x.size(0), -1)
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std

    def decode(self, z):
        x = self.fc_dec(z)
        x = x.view(-1, 256, 4, 4)
        x = self.decoder(x)
        return x

    def forward(self, x):
        mu, logvar = self.encode(x)

        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z)

        # Returns reconstructed image + latent parameters
        return x_recon, mu, logvar



In [5]:
latent_dim = 512
vae = VAE(latent_dim=latent_dim)
vae.load_state_dict(torch.load("vae_carracing.pth", map_location=device))
vae.to(device)
vae.eval()


for p in vae.parameters():
    p.requires_grad = False

In [6]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


In [7]:
class WorldModelTransformerMDN(nn.Module):
    def __init__(self, latent_dim, action_dim, d_model=256, nhead=8, num_layers=4, num_mixtures=5):
        super().__init__()

        self.num_mixtures = num_mixtures

        self.input_proj = nn.Linear(latent_dim + action_dim, d_model)
        self.pos_enc = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=512,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # MDN heads
        self.fc_mu = nn.Linear(d_model, latent_dim * num_mixtures)
        self.fc_sigma = nn.Linear(d_model, latent_dim * num_mixtures)
        self.fc_pi = nn.Linear(d_model, num_mixtures)

        # ✅ Reward head
        self.fc_reward = nn.Linear(d_model, 1)

    def forward(self, z, a):
        x = torch.cat([z, a], dim=-1)
        x = self.input_proj(x)
        x = self.pos_enc(x)
        h = self.transformer(x)

        context = h[:, -1, :].unsqueeze(1)

        batch_size, seq_len, _ = h.shape

        mu = self.fc_mu(h).view(batch_size, seq_len, -1, self.num_mixtures)
        sigma = torch.exp(self.fc_sigma(h)).view(batch_size, seq_len, -1, self.num_mixtures)
        pi = F.softmax(self.fc_pi(h), dim=-1)

        reward_pred = self.fc_reward(h).squeeze(-1)  # [batch, seq_len]

        return mu, sigma, pi, reward_pred, context

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

latent_dim = 512
action_dim = 3
num_mixtures = 5

model = WorldModelTransformerMDN(
    latent_dim=latent_dim,
    action_dim=action_dim,
    num_mixtures=num_mixtures
)

checkpoint = torch.load("/content/world_model_mdn_checkpoint (1).pth", map_location=device)

model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print("World model loaded successfully.")

World model loaded successfully.


In [9]:
!pip install cma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.3/328.3 kB 11.1 MB/s eta 0:00:00


In [10]:

!pip install gymnasium
!pip install swig
!pip install gymnasium[box2d]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 46.5 MB/s eta 0:00:00


In [ ]:
import gymnasium as gym
import cv2
import numpy as np
import torch
import torch.nn as nn
import cma
import os
import pickle

latent_dim = 512
mdn_dim = 256
action_dim = 3
sequence_len = 16

input_dim = latent_dim + mdn_dim

# CMA checkpoint path
cma_checkpoint = "/content/drive/MyDrive/cma_state.pkl"

class Controller(nn.Module):
    def __init__(self, input_dim, action_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, action_dim)

    def forward(self, x):
        return torch.tanh(self.linear(x))


def get_param_dim(input_dim, action_dim):
    return input_dim * action_dim + action_dim


def set_controller_params(controller, flat_params):

    weight_shape = controller.linear.weight.shape
    bias_shape = controller.linear.bias.shape

    num_weight = np.prod(weight_shape)

    weight = flat_params[:num_weight].reshape(weight_shape)
    bias = flat_params[num_weight:num_weight + np.prod(bias_shape)]

    with torch.no_grad():
        controller.linear.weight.copy_(torch.tensor(weight, dtype=torch.float32))
        controller.linear.bias.copy_(torch.tensor(bias, dtype=torch.float32))


param_dim = get_param_dim(input_dim, action_dim)
controller = Controller(input_dim, action_dim).to(device)

if os.path.exists(cma_checkpoint):

    print("Resuming CMA-ES from checkpoint...")

    with open(cma_checkpoint, "rb") as f:
        es = pickle.load(f)

    generation = es.countiter

else:

    print("Starting new CMA-ES training...")

    es = cma.CMAEvolutionStrategy(
        np.zeros(param_dim),
        0.5,
        {'popsize': 20, 'maxiter': 300}
    )

    generation = 0


def evaluate_controller(flat_params, controller, episodes=3):

    set_controller_params(controller, flat_params)

    total_reward = 0

    for ep in range(episodes):

        env = gym.make("CarRacing-v3", render_mode="rgb_array")

        obs, _ = env.reset()

        z_seq = []
        a_seq = []

        done = False
        ep_reward = 0

        while not done:

            obs_resized = cv2.resize(obs, (64, 64))

            obs_tensor = torch.tensor(obs_resized, dtype=torch.float32, device=device)
            obs_tensor = obs_tensor.permute(2, 0, 1).unsqueeze(0) / 255.0

            with torch.no_grad():
                mu, _ = vae.encode(obs_tensor)

            z = mu

            if len(z_seq) == 0:
                context = torch.zeros((1, mdn_dim), device=device)
            else:
                z_stack = torch.stack(z_seq).unsqueeze(0)
                a_stack = torch.stack(a_seq).unsqueeze(0)

                with torch.no_grad():
                    _, _, _, _, context = model(z_stack, a_stack)

                context = context.squeeze(0)
            controller_input = torch.cat([context, z], dim=1)
            with torch.no_grad():
                action = controller(controller_input).squeeze(0)

            action_np = action.cpu().numpy()
            obs, reward, terminated, truncated, _ = env.step(action_np)
            ep_reward += reward

            z_seq.append(z.squeeze(0))
            a_seq.append(action)

            if len(z_seq) > sequence_len:
                z_seq.pop(0)
                a_seq.pop(0)

            done = terminated or truncated
        env.close()

        total_reward += ep_reward
    return total_reward / episodes


def objective(flat_params):

    avg_reward = evaluate_controller(flat_params, controller)

    return -avg_reward

while not es.stop():

    generation += 1
    solutions = es.ask()
    fitnesses = [objective(s) for s in solutions]
    best_idx = np.argmin(fitnesses)
    best_gen_reward = -fitnesses[best_idx]
    print(f"Generation {generation}: Best Average Reward = {best_gen_reward:.2f}")
    es.tell(solutions, fitnesses)

    es.disp()

    # Save every 5 generations
    if generation % 5 == 0:

        save_path = f"/content/drive/MyDrive/controller_params_gen_{generation}.npy"
        np.save(save_path, solutions[best_idx])
        print(f"Saved controller parameters for generation {generation}")

        with open(cma_checkpoint, "wb") as f:
            pickle.dump(es, f)

        print("Checkpoint saved. Training can resume safely.")

best_params = es.result.xbest
np.save("/content/drive/MyDrive/controller_params_final.npy", best_params)

print("Controller training complete.")

Resuming CMA-ES from checkpoint...
Generation 21: Best Average Reward = 685.60
   21    420 -6.856008423585939e+02 1.0e+00 4.74e-01  5e-01  5e-01 247:44.7
Generation 22: Best Average Reward = 732.73
   22    440 -7.327272555457315e+02 1.0e+00 4.73e-01  5e-01  5e-01 258:50.5
Generation 23: Best Average Reward = 765.37
   23    460 -7.653719957406696e+02 1.0e+00 4.72e-01  5e-01  5e-01 270:39.2
Generation 24: Best Average Reward = 869.47
   24    480 -8.694661656853195e+02 1.0e+00 4.71e-01  5e-01  5e-01 282:20.7
Generation 25: Best Average Reward = 808.77
   25    500 -8.087702234849698e+02 1.0e+00 4.70e-01  5e-01  5e-01 294:22.2
Saved controller parameters for generation 25
Checkpoint saved. Training can resume safely.
Generation 26: Best Average Reward = 801.07
   26    520 -8.010657604808597e+02 1.0e+00 4.69e-01  5e-01  5e-01 306:28.8
Generation 27: Best Average Reward = 855.70
   27    540 -8.557046826478226e+02 1.0e+00 4.68e-01  5e-01  5e-01 318:35.1
Generation 28: Best Average Rewar

In [ ]:

best_params = es.result.xbest
np.save("/content/drive/MyDrive/controller_params_final.npy", best_params)
print("Controller training complete. Model saved.")